[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch02-ml-systems/01-dinh-luat-sat-va-diem-nghen.ipynb)

# Định luật sắt và điểm nghẽn, trên phần cứng của sách

Bài 2 dùng một công thức để trả lời câu hỏi "tải công việc này sẽ chạm rào cản nào trước": định luật
sắt (iron law) của hệ thống học máy. Nó tách thời gian thực thi thành ba phần: di chuyển dữ liệu, tính
toán, và phụ trội cố định (fixed overhead). Nguyên lý điểm nghẽn cho biết phần nào mới quan trọng.

Sổ tay này viết định luật sắt thành một hàm Python, rồi dùng nó với đúng các con số phần cứng chương
đưa ra. Không có phép đo nào ở đây; sổ tay 04 sẽ đo cùng nguyên lý đó trên CPU của bạn.

**Bạn sẽ làm:**
1. viết định luật sắt ở dạng cộng và dạng lấy giá trị lớn nhất;
2. làm lại Napkin Math 1.2: ResNet-50 trên một bộ tăng tốc (accelerator) trung tâm dữ liệu và trên
   một NPU di động;
3. tính điểm cân bằng máy (machine balance point), vẽ biểu đồ roofline, và tìm batch đưa ResNet-50 qua
   điểm gãy (ridge point);
4. xem một bộ xử lý nhanh hơn mua được gì cho một tải nghẽn ở bộ nhớ (memory-bound);
5. tính thời gian sinh một token của GPT-2 và Llama, kiểu Bandwidth Hog;
6. cộng dồn tường bộ nhớ qua các năm, rồi áp định luật Amdahl lên đường ống chụp ảnh.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

## 1. Định luật sắt: ba số hạng, cộng hay lấy giá trị lớn nhất

$$T = \frac{D_\text{vol}}{\text{BW}} + \frac{O}{R_\text{peak}\cdot\eta_\text{hw}} + L_\text{lat}$$

* $D_\text{vol}/\text{BW}$: khối lượng dữ liệu chia cho băng thông (bandwidth), thời gian di chuyển byte;
* $O/(R_\text{peak}\cdot\eta_\text{hw})$: số phép toán chia cho tích của thông lượng đỉnh và hiệu suất phần
  cứng, thời gian tính toán thực sự;
* $L_\text{lat}$: phụ trội cố định, như thời gian khởi chạy kernel.

Với một lần suy luận (inference) đơn lẻ, ba chi phí được trả lần lượt nên chúng cộng lại. Bộ tăng tốc
hiện đại thì cho việc nạp lô thứ $n+1$ chạy chồng lên việc tính lô thứ $n$. Khi các tầng chồng lấn hoàn
toàn, tổng biến thành một phép lấy giá trị lớn nhất:

$$T_\text{bottleneck} = \max\left(\frac{D_\text{vol}}{\text{BW}},\ \frac{O}{R_\text{peak}\cdot\eta_\text{hw}},\ T_\text{network}\right) + L_\text{lat}$$

Quy tắc của sách: dạng cộng cho độ trễ (latency) của một tác vụ, dạng lớn nhất cho thông lượng
(throughput) của một dòng tác vụ liên tục.

In [ ]:
MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # thứ tự màu cố định, như các sổ tay khác


def dinh_luat_sat(D, BW, O, R, eta=1.0, L=0.0, T_mang=0.0):
    """Ba số hạng của định luật sắt, tính bằng giây, và hai cách gộp chúng."""
    t_du_lieu = D / BW         # di chuyển byte
    t_tinh = O / (R * eta)     # tính toán thực sự
    return {
        "dữ liệu": t_du_lieu,
        "tính toán": t_tinh,
        "cộng": t_du_lieu + t_tinh + T_mang + L,             # một tác vụ, trả lần lượt
        "lớn nhất": max(t_du_lieu, t_tinh, T_mang) + L,      # dòng tác vụ, chồng lấn hoàn toàn
        "điểm nghẽn": max([("bộ nhớ", t_du_lieu), ("tính toán", t_tinh), ("mạng", T_mang)],
                          key=lambda cap: cap[1])[0],
    }


# một ví dụ đồ chơi: 1 GB dữ liệu qua 100 GB/s, 1 TFLOP trên máy 100 TFLOP/s
vd = dinh_luat_sat(D=1e9, BW=100e9, O=1e12, R=100e12, L=1e-3)
print({k: (f"{v * 1e3:.1f} ms" if isinstance(v, float) else v) for k, v in vd.items()})

## 2. ResNet-50 trên cloud và trên điện thoại (Napkin Math 1.2)

ResNet-50 là mô hình đại diện (lighthouse model) kiểu Compute Beast của chương: khoảng 8.2 GFLOP mỗi
ảnh, 25.6 triệu tham số, 102.4 MB ở FP32. Từ hai con số cuối suy ra mỗi tham số FP32 chiếm 4 byte, nên
FP16 chiếm 2 byte và INT8 chiếm 1 byte.

Napkin Math 1.2 đặt mô hình đó lên hai nền tảng, ở batch 1:

* một bộ tăng tốc trung tâm dữ liệu chạy FP16: 312 TFLOP/s, băng thông bộ nhớ 2.04 TB/s;
* một NPU di động cao cấp chạy INT8: khoảng 35 TOPS, khoảng 51.2 GB/s.

Byte phải di chuyển gồm trọng số cộng giá trị kích hoạt (activation): sách ghi $5.63\times10^{7}$ byte cho
bản FP16 và $2.82\times10^{7}$ byte cho bản INT8.

**Dự đoán:** ở batch 1, trên bộ tăng tốc 312 TFLOP/s, ResNet-50 nghẽn ở tính toán hay ở bộ nhớ? Còn
trên NPU di động?

In [ ]:
O_RESNET = sach(8.2, nguon="8.2 GFLOP") * 1e9          # FLOP mỗi ảnh
THAM_SO_RESNET = sach(25.6e6, nguon="25.6 triệu tham số")
MB_FP32 = sach(102.4, nguon="102.4 MB")
BYTE_FP32 = MB_FP32 * 1e6 / THAM_SO_RESNET
assert round(BYTE_FP32, 9) == 4.0                       # 4 byte mỗi tham số FP32
BYTE_FP16, BYTE_INT8 = BYTE_FP32 / 2, BYTE_FP32 / 4
theo_sach("trọng số FP16 (MB)", THAM_SO_RESNET * BYTE_FP16 / 1e6, sach=51.2, trich="51.2 MB FP16")
theo_sach("trọng số INT8 (MB)", THAM_SO_RESNET * BYTE_INT8 / 1e6, sach=25.6, trich="25.6 MB (INT8 quantized)")

# (a) bộ tăng tốc trung tâm dữ liệu, batch 1, FP16
R_CLOUD = sach(312, trich="312 TFLOP/s (FP16)") * 1e12
BW_CLOUD = sach(2.04, trich="Memory bandwidth: 2.04 TB/s") * 1e12
D_CLOUD = sach(5.63, trich=r"5.63 \times 10^{7}") * 1e7   # byte: trọng số FP16 cộng giá trị kích hoạt
cloud = dinh_luat_sat(D_CLOUD, BW_CLOUD, O_RESNET, R_CLOUD)
theo_sach("cloud: thời gian tính (ms)", 1e3 * cloud["tính toán"], sach=0.026, trich="= 0.026 ms")
theo_sach("cloud: thời gian bộ nhớ (ms)", 1e3 * cloud["dữ liệu"], sach=0.028, trich="= 0.028 ms")
theo_sach("cloud: bộ nhớ chậm hơn tính toán (lần)", cloud["dữ liệu"] / cloud["tính toán"], sach=1.1,
          trich="1.1× slower than compute")
theo_sach("FLOP trên mỗi byte di chuyển", O_RESNET / D_CLOUD, sach=146, trich="146 FLOP/byte")
print(f"điểm nghẽn trên cloud: {cloud['điểm nghẽn']}; "
      f"giá trị kích hoạt suy ra: {(D_CLOUD - THAM_SO_RESNET * BYTE_FP16) / 1e6:.1f} MB")

In [ ]:
# (b) NPU di động, batch 1, INT8
R_MOBILE = sach(35, trich="~35 TOPS (INT8)", tol=0.5) * 1e12
BW_MOBILE = sach(51.2, trich="~51.2 GB/s (LPDDR5)", tol=0.05) * 1e9
D_MOBILE = sach(2.82, trich=r"2.82 \times 10^{7}") * 1e7
mobile = dinh_luat_sat(D_MOBILE, BW_MOBILE, O_RESNET, R_MOBILE)
theo_sach("mobile: thời gian tính (ms)", 1e3 * mobile["tính toán"], sach=0.23, trich="= 0.23 ms")
theo_sach("mobile: thời gian bộ nhớ (ms)", 1e3 * mobile["dữ liệu"], sach=0.55, trich="= 0.55 ms")
print(f"điểm nghẽn trên mobile: {mobile['điểm nghẽn']}")

SACH_LAN = sach(2.3, trich="2.3× slower than compute")
print(f"mobile: bộ nhớ chậm hơn tính toán {mobile['dữ liệu'] / mobile['tính toán']:.3f} lần "
      f"(từ hai số đã làm tròn trong sách); sách ghi {SACH_LAN}×")

Cả hai đều nghẽn ở bộ nhớ, đúng kết luận của chương. Trên bộ tăng tốc, khoảng cách rất mỏng: bộ nhớ chỉ
chậm hơn tính toán khoảng 1.1 lần.

Một chỗ không khép hẳn: tính từ $2.82\times10^{7}$ byte, tỉ số của mobile là 2.35, trong khi sách ghi
2.3×. Bản INT8 dùng đúng một nửa số byte của bản FP16, tức $5.63\times10^{7}/2 = 2.815\times10^{7}$, mà
sách làm tròn thành 2.82. Với số chưa làm tròn, tỉ số là 2.347, và làm tròn xuống 2.3. Sổ tay chỉ in cả
hai, không khẳng định bên nào.

So sánh hai nền tảng: băng thông chênh nhau bao nhiêu, và thời gian suy luận chênh nhau bao nhiêu?

In [ ]:
lan_bang_thong = BW_CLOUD / BW_MOBILE
theo_sach("băng thông cloud so với mobile (lần)", lan_bang_thong, sach=39.8, trich="51.2 GB/s = 39.8×")
# cả hai đều nghẽn ở bộ nhớ, nên thời gian tỉ lệ với byte chia băng thông; INT8 chở nửa số byte của FP16
lan_suy_luan = lan_bang_thong * (BYTE_INT8 / BYTE_FP16)
theo_sach("suy luận nhanh hơn trên cloud (lần)", lan_suy_luan, sach=19.9, trich="19.9× faster inference")
lan_tinh = R_CLOUD / R_MOBILE
print(f"thông lượng đỉnh chênh {lan_tinh:.1f} lần, nhưng không phải con số quyết định")
print(f"tỉ số trực tiếp của hai thời gian bộ nhớ (dùng 2.82 đã làm tròn): "
      f"{mobile['dữ liệu'] / cloud['dữ liệu']:.2f} lần")

## 3. Điểm cân bằng máy và biểu đồ roofline

Chia số phép toán cho số byte di chuyển, ta có mật độ tính toán (arithmetic intensity) của tải, tính bằng
FLOP mỗi byte. Chia thông lượng đỉnh cho băng thông, ta có điểm cân bằng máy $R_\text{peak}/\text{BW}$.
Tải có mật độ dưới điểm đó thì nghẽn ở bộ nhớ, trên điểm đó thì nghẽn ở tính toán (compute-bound).

Từ định luật sắt suy ra ngay một đẳng thức: $T_\text{mem}/T_\text{comp}$ đúng bằng điểm cân bằng máy chia
cho mật độ tính toán.

In [ ]:
diem_gay_cloud = R_CLOUD / BW_CLOUD
diem_gay_mobile = R_MOBILE / BW_MOBILE
mat_do_cloud = O_RESNET / D_CLOUD
mat_do_mobile = O_RESNET / D_MOBILE
print(f"điểm cân bằng máy: cloud {diem_gay_cloud:.1f} FLOP/byte, mobile {diem_gay_mobile:.1f} phép/byte")
print(f"mật độ tính toán của ResNet-50, batch 1: FP16 {mat_do_cloud:.1f}, INT8 {mat_do_mobile:.1f}")
for ten, kq, gay, md in (("cloud", cloud, diem_gay_cloud, mat_do_cloud),
                         ("mobile", mobile, diem_gay_mobile, mat_do_mobile)):
    assert math.isclose(kq["dữ liệu"] / kq["tính toán"], gay / md)
    print(f"{ten}: T_mem/T_comp = {kq['dữ liệu'] / kq['tính toán']:.3f} = điểm gãy / mật độ")

Biểu đồ roofline vẽ thông lượng đạt được tối đa theo mật độ tính toán: $\min(R_\text{peak},\ I\cdot\text{BW})$.
Bên trái điểm gãy là mái dốc của băng thông, bên phải là mái phẳng của thông lượng đỉnh.

Batch đẩy một điểm sang phải. Để thấy nó đẩy bao xa, sổ tay **giả định** một cách đếm byte đơn giản:
mỗi batch đọc trọng số một lần, còn giá trị kích hoạt tỉ lệ với số ảnh. Phần kích hoạt của một ảnh lấy
từ chính sách: $5.63\times10^{7}$ byte trừ 51.2 MB trọng số FP16.

**Dự đoán:** cần batch bao nhiêu để ResNet-50 vượt điểm gãy trên cloud? Trên điện thoại?

In [ ]:
W_CLOUD, W_MOBILE = THAM_SO_RESNET * BYTE_FP16, THAM_SO_RESNET * BYTE_INT8
A_CLOUD, A_MOBILE = D_CLOUD - W_CLOUD, D_MOBILE - W_MOBILE   # byte kích hoạt mỗi ảnh


def mat_do(batch, W, A):
    """FLOP mỗi byte khi một batch đọc trọng số một lần (giả định của sổ tay)."""
    return batch * O_RESNET / (W + batch * A)


def batch_vuot(W, A, diem_gay):
    """Batch nhỏ nhất có mật độ vượt điểm gãy, hoặc None nếu không bao giờ vượt."""
    du = O_RESNET / diem_gay - A
    return None if du <= 0 else math.floor(W / du) + 1


print(f"byte kích hoạt mỗi ảnh: FP16 {A_CLOUD / 1e6:.2f} MB, INT8 {A_MOBILE / 1e6:.2f} MB")
for he_so in (1, 3, 10):
    b_c = batch_vuot(W_CLOUD, he_so * A_CLOUD, diem_gay_cloud)
    b_m = batch_vuot(W_MOBILE, he_so * A_MOBILE, diem_gay_mobile)
    b_m = "không bao giờ" if b_m is None else f"từ batch {b_m}"
    print(f"kích hoạt × {he_so:>2}: cloud vượt điểm gãy từ batch {b_c}, mobile {b_m}"
          f"  (trần mật độ khi batch rất lớn: {O_RESNET / (he_so * A_CLOUD):,.0f} và "
          f"{O_RESNET / (he_so * A_MOBILE):,.0f})")

In [ ]:
I = np.logspace(0, 4, 400)
fig, ax = plt.subplots(figsize=(8.5, 4.8))
for ten, R, BW, W, A, mau, gay in (("bộ tăng tốc cloud, FP16", R_CLOUD, BW_CLOUD, W_CLOUD, A_CLOUD, MAU[0], diem_gay_cloud),
                                    ("NPU di động, INT8", R_MOBILE, BW_MOBILE, W_MOBILE, A_MOBILE, MAU[1], diem_gay_mobile)):
    ax.loglog(I, np.minimum(R, I * BW) / 1e12, color=mau, lw=2, label=ten)
    ax.plot([gay], [R / 1e12], "s", color=mau, ms=7)
    ax.annotate(f"điểm gãy {gay:.0f}", (gay, R / 1e12), xytext=(-20, 16), textcoords="offset points",
                fontsize=8, color="0.25", arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
    cac_batch = np.array([1, 2, 4, 16, 64])
    md = mat_do(cac_batch, W, A)
    ax.plot(md, np.minimum(R, md * BW) / 1e12, "o", color=mau, ms=8, mec="white", mew=1.5)
    ax.annotate(f"batch 1: {md[0]:.0f}", (md[0], md[0] * BW / 1e12), xytext=(-110, 8) if mau == MAU[0] else (-80, 14),
                textcoords="offset points", fontsize=8, color="0.25",
                arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax.set_ylim(None, 3000)
ax.set_xlabel("mật độ tính toán (phép tính mỗi byte di chuyển)")
ax.set_ylabel("thông lượng tối đa (nghìn tỉ phép/s)")
ax.set_title("Roofline của hai nền tảng; chấm tròn: ResNet-50 ở batch 1, 2, 4, 16, 64")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

Với cách đếm của sách, giá trị kích hoạt của một ảnh chỉ chiếm khoảng một phần mười số byte, nên chỉ cần
batch 2 trên cloud và batch 3 trên điện thoại là ResNet-50 đã qua điểm gãy. Đó là một con số rất nhạy.
Nếu lưu lượng kích hoạt thật lớn gấp 10 lần (tensor trung gian được ghi ra rồi đọc lại), điểm vượt trên
cloud lùi tới batch 20, còn trên NPU thì không bao giờ tới: trần mật độ khi đó thấp hơn điểm gãy. Đây là
lý do chương nói điểm chuyển giữa hai chế độ phụ thuộc kiến trúc và cách hiện thực, và phải đo trên đúng
môi trường chạy.

## 4. Cây cầu hẹp: bộ xử lý nhanh hơn mua được gì?

Mở rộng đường cao tốc không giúp dòng xe đang bị một cây cầu hẹp chặn lại. Với ResNet-50 trên NPU di
động (nghẽn ở bộ nhớ), hãy nhân thông lượng đỉnh, hoặc băng thông, lên $k$ lần.

**Dự đoán:** gấp đôi $R_\text{peak}$ làm suy luận nhanh lên bao nhiêu? Gấp đôi băng thông thì sao?

In [ ]:
cac_k = np.linspace(1, 6, 51)


def tang_toc(k_R=1.0, k_BW=1.0, dang="lớn nhất"):
    goc = dinh_luat_sat(D_MOBILE, BW_MOBILE, O_RESNET, R_MOBILE)[dang]
    moi = dinh_luat_sat(D_MOBILE, BW_MOBILE * k_BW, O_RESNET, R_MOBILE * k_R)[dang]
    return goc / moi


for k in (2, 4):
    print(f"R × {k}: nhanh hơn {tang_toc(k_R=k):.2f} lần (dạng lớn nhất), {tang_toc(k_R=k, dang='cộng'):.2f} lần (dạng cộng)")
    print(f"BW × {k}: nhanh hơn {tang_toc(k_BW=k):.2f} lần (dạng lớn nhất), {tang_toc(k_BW=k, dang='cộng'):.2f} lần (dạng cộng)")
assert tang_toc(k_R=4) == 1.0   # dạng lớn nhất: thêm tính toán không giúp gì khi nghẽn ở bộ nhớ

def thoi_gian_ms(k_R=1.0, k_BW=1.0, dang="lớn nhất"):
    return 1e3 * dinh_luat_sat(D_MOBILE, BW_MOBILE * k_BW, O_RESNET, R_MOBILE * k_R)[dang]


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
for ax, khoa, tieu_de in ((ax1, "k_R", "Nhân thông lượng đỉnh R_peak lên k lần"), (ax2, "k_BW", "Nhân băng thông BW lên k lần")):
    for dang, kieu, mau in (("cộng", "--", MAU[1]), ("lớn nhất", "-", MAU[0])):
        ax.plot(cac_k, [thoi_gian_ms(**{khoa: k}, dang=dang) for k in cac_k], kieu, color=mau, lw=2,
                label=f"dạng {dang}")
    ax.set_xlabel("hệ số k")
    ax.set_title(tieu_de, fontsize=10)
k_gay = diem_gay_mobile / mat_do_mobile
ax2.axvline(k_gay, color="0.5", lw=0.8, ls=":")
ax2.text(k_gay + 0.08, 0.66, "từ đây nghẽn\nở tính toán", fontsize=8, color="0.35", va="top")
ax1.set_ylim(0, 0.85)
ax1.set_ylabel("thời gian mỗi ảnh (ms)")
ax1.legend(loc="lower left", fontsize=8)
fig.suptitle("ResNet-50 trên NPU di động, batch 1", fontsize=10)
plt.tight_layout()
plt.show()

Ở dạng lớn nhất, nhân thông lượng đỉnh lên bao nhiêu lần cũng không nhanh thêm chút nào: đường xanh nằm
ngang. Ở dạng cộng, nó chỉ gọt dần số hạng tính toán vốn đã nhỏ, và không bao giờ xuống dưới thời gian bộ
nhớ. Nhân băng thông thì có ích ngay, cho tới khi số hạng tính toán
thành số hạng lớn nhất: từ đó tải chuyển sang nghẽn ở tính toán. Vì thế kỹ sư phải xác định số hạng chi
phối trước, rồi mới tối ưu, và vì thế các kỹ thuật giảm byte như hạ độ chính xác số (numerical precision)
thường thắng việc mua thêm FLOP đỉnh mỗi giây.

## 5. Bandwidth Hog: sinh một token của GPT-2 và Llama

Sinh văn bản tự hồi quy (autoregressive) tạo từng token một. Khi giải mã với lô nhỏ, mỗi token có thể đòi
chuyển gần hết hoặc toàn bộ tham số qua bộ nhớ: 1.5 tỉ tham số với GPT-2, từ 7 tỉ đến 70 tỉ với Llama.

Sổ tay **giả định** mỗi token đọc toàn bộ trọng số một lần, và tốn khoảng 2 phép tính cho mỗi tham số (một
phép nhân-cộng với mỗi trọng số). Bộ nhớ đệm KV và phụ trội cố định bị bỏ qua, nên mọi con số dưới đây là
**cận dưới**.

**Dự đoán:** trên bộ tăng tốc 312 TFLOP/s, thời gian một token của Llama 7B do tính toán hay do băng thông
quyết định, và chênh nhau cỡ bao nhiêu lần?

In [ ]:
MO_HINH_LLM = {
    "GPT-2": sach(1.5e9, nguon="1.5 tỉ / 7–70 tỉ"),
    "Llama 7B": sach(7, nguon="1.5 tỉ / 7–70 tỉ") * 1e9,
    "Llama 70B": sach(70e9, nguon="1.5 tỉ / 7–70 tỉ"),
}
TOKEN_SACH = (sach(20, nguon="20–100 ms"), sach(100, nguon="20–100 ms"))
RAM_MOBILE = (sach(8, nguon="8–16 GB, 128 GB–1 TB, 3–5 W, $999+"), sach(16, nguon="8–16 GB, 128 GB–1 TB, 3–5 W, $999+"))

print(f"{'mô hình':<10} {'nền tảng':<22} {'trọng số':>9} {'T_mem':>9} {'T_comp':>9} {'chênh':>6} {'token/s tối đa':>15}")
for ten, n in MO_HINH_LLM.items():
    for nen_tang, R, BW, b in (("cloud, FP16", R_CLOUD, BW_CLOUD, BYTE_FP16), ("NPU di động, INT8", R_MOBILE, BW_MOBILE, BYTE_INT8)):
        kq = dinh_luat_sat(D=n * b, BW=BW, O=2 * n, R=R)
        ghi_chu = "" if nen_tang.startswith("cloud") or n * b <= RAM_MOBILE[1] * 1e9 else "  (không vừa 8–16 GB)"
        print(f"{ten:<10} {nen_tang:<22} {n * b / 1e9:>6.1f} GB {1e3 * kq['dữ liệu']:>6.1f} ms {1e3 * kq['tính toán']:>6.3f} ms"
              f" {kq['dữ liệu'] / kq['tính toán']:>5.0f}× {1 / kq['lớn nhất']:>13.0f}{ghi_chu}")
print(f"\nmật độ tính toán khi giải mã batch 1: FP16 {2 / BYTE_FP16:.0f} FLOP/byte, INT8 {2 / BYTE_INT8:.0f} phép/byte")
print(f"bảng độ trễ của chương: sinh một token mất {TOKEN_SACH[0]}–{TOKEN_SACH[1]} ms")

Mật độ tính toán khi giải mã ở batch 1 chỉ khoảng 1 FLOP mỗi byte, trong khi điểm gãy của bộ tăng tốc là
khoảng 153. Vì thế số hạng băng thông lớn hơn số hạng tính toán khoảng 150 lần, và thêm FLOP gần như vô
nghĩa với tải này. Đó là định nghĩa của Bandwidth Hog.

Cách thoát mà chương nêu là tăng tái sử dụng dữ liệu: phục vụ $B$ yêu cầu cùng lúc thì một lần đọc trọng
số phục vụ $B$ token, và mật độ tăng gần tỉ lệ với $B$:

In [ ]:
n7 = MO_HINH_LLM["Llama 7B"]
cac_batch = [1, 8, 32, 128, 512]
for b in cac_batch:
    kq = dinh_luat_sat(D=n7 * BYTE_FP16, BW=BW_CLOUD, O=2 * n7 * b, R=R_CLOUD)
    nghen = f"nghẽn ở {kq['điểm nghẽn']}"
    print(f"batch {b:>3}: mật độ {2 * b / BYTE_FP16:>4.0f} FLOP/byte, {nghen:<18} "
          f"{b / kq['lớn nhất']:>8,.0f} token/s tổng, {1e3 * kq['lớn nhất']:.1f} ms mỗi bước")
print(f"vượt điểm gãy từ batch khoảng {diem_gay_cloud * BYTE_FP16 / 2:.0f} (bỏ qua bộ nhớ đệm KV)")

## 6. Tường bộ nhớ cộng dồn

Năng lực tính toán gấp đôi khoảng mỗi 18 tháng, còn băng thông bộ nhớ chỉ khá lên khoảng 20 % mỗi năm.
Sách quy cả hai về hệ số tăng trưởng hằng năm và lấy tỉ số.

In [ ]:
THANG_GAP_DOI = sach(18, nguon="18 tháng")
he_so_tinh = 2 ** (12 / THANG_GAP_DOI)
he_so_bw = 1 + sach(0.2, nguon="20 %")
theo_sach("hệ số tăng tính toán mỗi năm", he_so_tinh, sach=1.6, nguon="1.6 / 1.2")
theo_sach("hệ số tăng băng thông mỗi năm", he_so_bw, sach=1.2, nguon="1.6 / 1.2")
TANG_TINH, TANG_BW = sach(1.6, nguon="1.6 / 1.2"), sach(1.2, nguon="1.6 / 1.2")
TI_SO = TANG_TINH / TANG_BW
theo_sach("tỉ số tăng trưởng", TI_SO, sach=1.33, nguon="1.6 / 1.2 ≈ 1.33", tol=0.005)
for nam in (1, 5, 10):
    print(f"sau {nam:>2} năm: tính toán × {TANG_TINH ** nam:6.1f}, băng thông × {TANG_BW ** nam:5.1f}, "
          f"khoảng cách × {TI_SO ** nam:5.1f}")

Tỉ số lớn hơn 1 có nghĩa là điểm cân bằng máy cứ dịch sang phải mỗi năm. Một tải có mật độ cố định, như
ResNet-50 ở batch 1, vì thế càng ngày càng nghẽn ở bộ nhớ nặng hơn. Đồ thị dưới đây **giả định** xu hướng
giữ nguyên và bắt đầu từ bộ tăng tốc ở phần 2; đó là một phép ngoại suy để thấy hình dạng, không phải dự
báo cho một con chip cụ thể.

In [ ]:
nam = np.arange(0, 11)
gay_theo_nam = diem_gay_cloud * TI_SO ** nam
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.semilogy(nam, gay_theo_nam, "o-", color=MAU[0], lw=2, label="điểm cân bằng máy (ngoại suy)")
ax.axhline(mat_do_cloud, color=MAU[1], lw=2, label="ResNet-50 batch 1, FP16")
ax.annotate(f"năm 0: {gay_theo_nam[0]:.0f}", (0, gay_theo_nam[0]), xytext=(10, 22), textcoords="offset points",
            fontsize=8, color="0.25", arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax.annotate(f"{TI_SO ** 10:.0f}× sau 10 năm: {gay_theo_nam[-1]:,.0f}", (10, gay_theo_nam[-1]), xytext=(-150, 0),
            textcoords="offset points", fontsize=8, color="0.25", va="center",
            arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax.text(10, mat_do_cloud * 1.12, f"{mat_do_cloud:.0f} FLOP/byte", fontsize=8, color="0.25", ha="right")
ax.set_ylim(100, 5000)
ax.set_yticks([100, 200, 500, 1000, 2000, 5000], ["100", "200", "500", "1,000", "2,000", "5,000"])
ax.set_xlabel("năm")
ax.set_ylabel("FLOP mỗi byte (thang log)")
ax.set_title("Khoảng cách tính toán / băng thông nhân 1.33 lần mỗi năm")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## 7. Định luật Amdahl trên đường ống chụp ảnh

Ngộ nhận thứ năm của chương: tối ưu mô hình sẽ tăng tốc hệ thống theo tỉ lệ tuyến tính. Khi bấm chụp, ảnh
đi qua 100 ms xử lý tín hiệu, 60 ms phân loại khung cảnh bằng học máy và 40 ms hậu xử lý. Định luật Amdahl
(Amdahl's law) cho tăng tốc toàn hệ thống khi phần $p$ được tăng tốc $s$ lần:

$$\text{Speedup} = \frac{1}{(1-p) + p/s}$$

**Dự đoán:** làm bộ phân loại nhanh gấp 10 lần thì cả lần chụp nhanh hơn bao nhiêu?

In [ ]:
TIN_HIEU = sach(100, nguon="100 + 60 + 40 = 200 ms")
PHAN_LOAI = sach(60, nguon="100 + 60 + 40 = 200 ms")
HAU_XU_LY = sach(40, nguon="100 + 60 + 40 = 200 ms")
tong = TIN_HIEU + PHAN_LOAI + HAU_XU_LY
theo_sach("tổng thời gian một lần chụp (ms)", tong, sach=200, nguon="100 + 60 + 40 = 200 ms")
p = PHAN_LOAI / tong


def amdahl(s):
    return 1 / ((1 - p) + p / s)


moi = TIN_HIEU + PHAN_LOAI / 10 + HAU_XU_LY
theo_sach("bộ phân loại nhanh gấp 10 (ms)", PHAN_LOAI / 10, sach=6, nguon="60 → 6 ms, 200 → 146 ms, 1.37×")
theo_sach("tổng sau khi tối ưu (ms)", moi, sach=146, nguon="60 → 6 ms, 200 → 146 ms, 1.37×")
theo_sach("tăng tốc toàn hệ thống", amdahl(10), sach=1.37, nguon="60 → 6 ms, 200 → 146 ms, 1.37×")
theo_sach("trần khi bỏ hẳn học máy", amdahl(math.inf), sach=1.43, nguon="1.43×")
theo_sach("phần không được tối ưu (%)", 100 * (1 - p), sach=70, nguon="70 %")
assert math.isclose(amdahl(10), tong / moi)

In [ ]:
cac_s = np.logspace(0, 3, 200)
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.semilogx(cac_s, amdahl(cac_s), color=MAU[0], lw=2)
ax.axhline(amdahl(math.inf), color="0.5", lw=0.8, ls="--")
ax.text(1.1, amdahl(math.inf) + 0.01, f"trần {amdahl(math.inf):.2f}×: bỏ hẳn học máy", fontsize=8, color="0.35")
ax.plot([10], [amdahl(10)], "o", color=MAU[1], ms=8)
ax.annotate(f"phân loại nhanh gấp 10: {amdahl(10):.2f}×", (10, amdahl(10)), xytext=(14, -26),
            textcoords="offset points", fontsize=8, arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax.set_ylim(0.95, 1.5)
ax.set_xlabel("bộ phân loại nhanh hơn s lần")
ax.set_ylabel("cả lần chụp nhanh hơn (lần)")
ax.set_title("Định luật Amdahl: 70 % đường ống không đổi")
plt.tight_layout()
plt.show()

Đó là dạng cộng: một tấm ảnh đi qua ba chặng lần lượt. Giờ **giả định** ba chặng chạy trên ba khối xử lý
riêng và chồng lên nhau qua các tấm ảnh liên tiếp. Khi đó số ảnh mỗi giây theo dạng lớn nhất, do chặng
chậm nhất quyết định:

In [ ]:
for ten, t_pl in (("trước khi tối ưu", PHAN_LOAI), ("phân loại nhanh gấp 10", PHAN_LOAI / 10)):
    print(f"{ten:<24}: một tấm ảnh {TIN_HIEU + t_pl + HAU_XU_LY:5.0f} ms, "
          f"dòng ảnh liên tục {1000 / max(TIN_HIEU, t_pl, HAU_XU_LY):4.1f} ảnh/s")

Độ trễ của một tấm ảnh giảm từ 200 xuống 146 ms, còn thông lượng của dòng ảnh không đổi, vì chặng xử lý
tín hiệu 100 ms vẫn là chặng chậm nhất. Hai câu hỏi khác nhau dùng hai dạng khác nhau của cùng một định
luật, đúng như quy tắc kinh nghiệm của chương.

## Thử thêm

1. Ở phần 2, thay NPU di động bằng đầu trên của vùng mobile trong chương: 45 TOPS và 77 GB/s. Điểm cân
   bằng máy thay đổi thế nào, và ResNet-50 có còn nghẽn ở bộ nhớ ở batch 1 không? (Các dấu ✗ khi đó là
   điều được chờ đợi.)
2. Ở phần 5, lượng tử hoá Llama 7B xuống 4 bit (nửa byte mỗi tham số). Trên NPU di động, cận dưới của thời
   gian một token là bao nhiêu, và mô hình 70B có vừa 16 GB không?
3. Ở phần 7, giữ bộ phân loại 60 ms nhưng làm chặng xử lý tín hiệu nhanh gấp đôi. Độ trễ một tấm ảnh và
   thông lượng của dòng ảnh mỗi thứ đổi bao nhiêu? Vì sao kết quả ngược với lần tối ưu bộ phân loại?

## Tóm lại

* Định luật sắt cộng ba số hạng cho một tác vụ, và lấy số hạng lớn nhất cho một dòng tác vụ chồng lấn.
* ResNet-50 ở batch 1 nghẽn ở bộ nhớ cả trên bộ tăng tốc cloud lẫn NPU di động; tỉ số $T_\text{mem}/T_\text{comp}$
  chính là điểm cân bằng máy chia cho mật độ tính toán.
* Khi nghẽn ở bộ nhớ, thêm FLOP đỉnh không mua được gì; giảm byte hoặc tăng tái sử dụng (batch) mới có ích,
  và điểm vượt sang nghẽn ở tính toán rất nhạy với cách đếm byte.
* Giải mã mô hình ngôn ngữ ở batch 1 có mật độ khoảng 1 FLOP mỗi byte, nên băng thông quyết định thời gian
  mỗi token; tường bộ nhớ làm khoảng cách đó rộng thêm mỗi năm.
* Tối ưu một chặng chỉ giúp theo phần chặng đó chiếm (Amdahl), và không giúp thông lượng nếu chặng đó không
  phải chặng chậm nhất.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Hệ thống học máy, bốn môi trường triển khai](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch02-ml-systems/), dựng từ chương
[*ML Systems*](https://mlsysbook.ai/vol1/ml_systems/ml_systems.html) của sách.

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.